# 2. Datenbereinigung - Titanic Dataset

In diesem Notebook werden die Titanic-Daten für statistische Analysen und Machine-Learning-Modelle vorbereitet.

## Ziele
- Datensatz laden
- Fehlende Werte behandeln
- Unnötige oder problematische Spalten entfernen
- Kategorische Variablen kodieren
- Bereinigte Daten speichern

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
print('Bibliotheken erfolgreich importiert!')

Bibliotheken erfolgreich importiert!


## 2.1 Datensatz laden

In [2]:
data_path = Path('../data/raw/titanic.csv')
df = pd.read_csv(data_path)

print(f'Dataset geladen: {data_path}')
print(f'Zeilen: {df.shape[0]} | Spalten: {df.shape[1]}')
df.head()

Dataset geladen: ..\data\raw\titanic.csv
Zeilen: 891 | Spalten: 15


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2.2 Fehlende Werte und Spalten prüfen

In [3]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'fehlende_werte': missing,
    'prozent': missing_percent.round(2)
})

missing_summary[missing_summary['fehlende_werte'] > 0]

,fehlende_werte,prozent
deck,688,77.22
Age,177,19.87
Embarked,2,0.22
embark_town,2,0.22


## 2.3 Bereinigungsstrategie

Ich verwende die folgenden Schritte:
- `deck` entfernen, da sehr viele Werte fehlen
- `alive` entfernen, da diese Spalte die Zielvariable in Textform dupliziert
- `class` entfernen, da sie inhaltlich stark mit `pclass` überlappt
- Fehlende Werte in `age` mit dem Median ersetzen
- Fehlende Werte in `embarked` und `embark_town` mit dem Modus ersetzen

In [4]:
df_clean = df.copy()

# Spalten entfernen (falls vorhanden)
columns_to_drop = [col for col in ['deck', 'alive', 'class', 'Deck', 'Alive', 'Class'] if col in df_clean.columns]
df_clean = df_clean.drop(columns=columns_to_drop)

# Robuste Behandlung für unterschiedliche Schreibweisen
age_col = 'age' if 'age' in df_clean.columns else ('Age' if 'Age' in df_clean.columns else None)
embarked_col = 'embarked' if 'embarked' in df_clean.columns else ('Embarked' if 'Embarked' in df_clean.columns else None)
embark_town_col = 'embark_town' if 'embark_town' in df_clean.columns else ('Embark_town' if 'Embark_town' in df_clean.columns else None)

if age_col is not None:
    df_clean[age_col] = df_clean[age_col].fillna(df_clean[age_col].median())

if embarked_col is not None:
    df_clean[embarked_col] = df_clean[embarked_col].fillna(df_clean[embarked_col].mode()[0])

if embark_town_col is not None:
    df_clean[embark_town_col] = df_clean[embark_town_col].fillna(df_clean[embark_town_col].mode()[0])

print('Bereinigung abgeschlossen.')
print(f'Neue Form: {df_clean.shape}')
df_clean.isnull().sum().sort_values(ascending=False).head(10)

Bereinigung abgeschlossen.
Neue Form: (891, 12)


Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
who           0
adult_male    0
dtype: int64

## 2.4 Kategorische Variablen kodieren

Für Machine-Learning-Modelle werden kategoriale Spalten in numerische Form überführt.

In [7]:
# Redundanz entfernen: wenn Embarked vorhanden ist, embark_town nicht zusaetzlich encodieren
df_model_input = df_clean.copy()
if 'Embarked' in df_model_input.columns and 'embark_town' in df_model_input.columns:
    df_model_input = df_model_input.drop(columns=['embark_town'])
    print("Info: 'embark_town' wurde entfernt (redundant zu 'Embarked').")

categorical_columns = df_model_input.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print('Kategoriale Spalten:', categorical_columns)

df_model = pd.get_dummies(df_model_input, columns=categorical_columns, drop_first=True)
print(f'Modelldaten Form: {df_model.shape}')
df_model.head()

Info: 'embark_town' wurde entfernt (redundant zu 'Embarked').
Kategoriale Spalten: ['Sex', 'Embarked', 'who', 'adult_male', 'alone']
Modelldaten Form: (891, 13)


,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,who_man,who_woman,adult_male_True,alone_True
0,0,3,22.0,1,0,7.2500,True,False,True,True,False,True,False
1,1,1,38.0,1,0,71.2833,False,False,False,False,True,False,False
2,1,3,26.0,0,0,7.9250,False,False,True,False,True,False,True
3,1,1,35.0,1,0,53.1000,False,False,True,False,True,False,False
4,0,3,35.0,0,0,8.0500,True,False,True,True,False,True,True


## 2.5 Bereinigte Daten speichern

In [8]:
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

clean_output = processed_dir / 'titanic_cleaned.csv'
model_output = processed_dir / 'titanic_model_ready.csv'

df_clean.to_csv(clean_output, index=False)
df_model.to_csv(model_output, index=False)

print(f'Gespeichert: {clean_output}')
print(f'Gespeichert: {model_output}')

Gespeichert: ..\data\processed\titanic_cleaned.csv
Gespeichert: ..\data\processed\titanic_model_ready.csv
